In [ ]:
# !nvidia-smi

In [ ]:
# !pip install -U transformers accelerate

In [2]:
!pip install -U bitsandbytes peft trl datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 40.0 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 996.6/996.6 kB 58.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 31.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 62.1 MB/s  0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 23.0.1
    Uninstalling pyarrow-23.0.1:
      Successfully uninstalled pyarrow-23.0.1
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.5
    Uninstalling datasets-4.8.5:
      Successfully uninstalled datasets-4.8.5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [trl]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


In [3]:
# !pip install -U --force-reinstall pyarrow

In [4]:
import torch
import transformers
import accelerate
import bitsandbytes as bnb
import peft
import trl
import datasets

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("BitsAndBytes:", bnb.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("Datasets:", datasets.__version__)

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
Transformers: 5.21.0
Accelerate: 1.15.0
BitsAndBytes: 0.50.2
PEFT: 0.21.2
TRL: 1.17.1
Datasets: 5.1.0


In [5]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU is not enabled or PyTorch lacks CUDA support.")

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "Qwen/Qwen3-4B"

# 4-bit quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
# Load tokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer:
  print("Tokenizer loaded successfully!")
else:
  print("Tokenizer failed to load!")


# Load model onto GPU
new_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    dtype=torch.float16,
    device_map="auto",
)

print("Model loaded successfully!")
print("GPU:", torch.cuda.get_device_name(0))
print("Model memory:", round(
    torch.cuda.memory_allocated() / 1024**3, 2
), "GB")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (3 outputs left out)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model loaded successfully!
GPU: Tesla T4
Model memory: 2.49 GB


In [7]:
from peft import prepare_model_for_kbit_training

new_model.config.use_cache = False

new_model = prepare_model_for_kbit_training(
    new_model,
    use_gradient_checkpointing=True,
)

In [ ]:
# print("Model config dtype:", model.config.torch_dtype)
# print("Quantization config:", model.config.quantization_config)
# print("Tokenizer loaded:", tokenizer is not None)

In [8]:
text = "What is article 14?"
tokens = tokenizer.tokenize(text)
token_ids = tokenizer.encode(text)

print("Tokens:", tokens)
print("Token IDs:", token_ids)

Tokens: ['What', 'Ġis', 'Ġarticle', 'Ġ', '1', '4', '?']
Token IDs: [3838, 374, 4549, 220, 16, 19, 30]


In [13]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files = "/content/constitution_of_india.json",
    split = "train"
)

print(dataset)
print("Total records:", len(dataset))
print("First record:", dataset[0])
print("Dataset columns:", dataset.column_names)


Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['article', 'title', 'description'],
    num_rows: 465
})
Total records: 465
First record: {'article': 0, 'title': 'Preamble', 'description': 'WE, THE PEOPLE OF INDIA, having solemnly resolved to constitute India into a SOVEREIGN SOCIALIST SECULAR DEMOCRATIC REPUBLIC and to secure to all its citizens: JUSTICE, social, economic and political; LIBERTY of thought, expression, belief, faith and worship;\nEQUALITY of status and of opportunity; and to promote among them all FRATERNITY assuring the dignity of the individual and the unity and integrity of the Nation; IN OUR CONSTITUENT ASSEMBLY this twenty-sixth day of November, 1949, do HEREBY ADOPT, ENACT AND GIVE TO OURSELVES THIS CONSTITUTION'}
Dataset columns: ['article', 'title', 'description']


In [14]:
print(tokenizer.tokenize("What is the Preamble of India?"))

['What', 'Ġis', 'Ġthe', 'ĠP', 'reamble', 'Ġof', 'ĠIndia', '?']


In [15]:

def format_article(example):
    title = example["title"].strip()
    description = example["description"].strip()
    article_no = example["article"]

    if title.lower() == "preamble":
        question = "Explain the Preamble of the Constitution of India."
    else:
        question = (
            f"Explain Article {article_no}: {title} "
            "of the Constitution of India."
        )

    return {
        "text": f"### Question:\n{question}\n\n"
                f"### Answer:\n{description}"
    }

formatted_dataset = dataset.map(format_article)

print("Total records:", len(formatted_dataset))

Map:   0%|          | 0/465 [00:00<?, ? examples/s]

Total records: 465


In [18]:
for i, item in enumerate(formatted_dataset):
    text = item["text"]
    if "Article 14:" in text or "Article 13:" in text:
        print(f"\n--- SAMPLE {i} ---")
        print(text[:1000])


--- SAMPLE 13 ---
### Question:
Explain Article 13: Laws inconsistent with or in derogation of the fundamental rights of the Constitution of India.

### Answer:
(1) All laws in force in the territory of India immediately before the commencement of this Constitution, in so far as they are inconsistent with the provisions of this Part, shall, to the extent of such inconsistency, be void.

 

(2) The State shall not make any law which takes away or abridges the rights conferred by this Part and any law made in contravention of this clause shall, to the extent of the contravention, be void.

 

(3) In this article, unless the context otherwise requires,—

       (a) “law” includes any Ordinance, order, bye-law, rule, regulation, notification, custom or usage having in the territory of India the force of law;

       (b) “laws in force” includes laws passed or made by a Legislature or other competent authority in the territory of India before the commencement of this Constitution and not p

In [16]:
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig , prepare_model_for_kbit_training , get_peft_model
from datasets import load_dataset

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

training_args = SFTConfig(
    output_dir="./legal-advisor",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    logging_steps=10,
    report_to="none",
    loss_type="nll",
    fp16=False,
    bf16=False,
)


In [17]:
from peft import get_peft_model

new_model = get_peft_model(new_model, lora_config)

from collections import Counter

print(Counter(
    str(p.dtype)
    for p in new_model.parameters()
    if p.requires_grad
))

Counter({'torch.float32': 504})


In [ ]:
new_trainer = SFTTrainer(
    model=new_model,
    args=training_args,
    train_dataset=formatted_dataset,
    processing_class=tokenizer,
)

In [ ]:
new_trainer.train()

Step,Training Loss
10,1.849319
20,1.561240
30,1.335391
40,1.310356
50,1.281584
60,1.173827
70,1.166482
80,1.120352
90,1.124034
100,1.136020


TrainOutput(global_step=177, training_loss=1.1965236044199454, metrics={'train_runtime': 1943.6402, 'train_samples_per_second': 0.718, 'train_steps_per_second': 0.091, 'total_flos': 6924682703877120.0, 'train_loss': 1.1965236044199454, 'entropy': 1.0234985838131028, 'mean_token_accuracy': 0.7428374764870624, 'num_tokens': 314769.0, 'epoch': 3.0})

In [ ]:
from peft import get_peft_model

new_model = get_peft_model(new_model, lora_config)

from collections import Counter

print(Counter(
    str(p.dtype)
    for p in new_model.parameters()
    if p.requires_grad
))

Counter({'torch.float32': 504})


In [ ]:
from collections import Counter

print("Trainable parameter dtypes:")
print(Counter(
    str(p.dtype)
    for p in new_trainer.model.parameters()
    if p.requires_grad
))

print("\nExisting gradient dtypes:")
print(Counter(
    str(p.grad.dtype)
    for p in new_trainer.model.parameters()
    if p.grad is not None
))

Trainable parameter dtypes:
Counter({'torch.float32': 504})

Existing gradient dtypes:
Counter()


In [ ]:
new_trainer.save_model("./legal-advisor-final-1")
tokenizer.save_pretrained("./legal-advisor-final-1")

('./legal-advisor-final-1/tokenizer_config.json',
 './legal-advisor-final-1/chat_template.jinja',
 './legal-advisor-final-1/tokenizer.json')

In [ ]:
import os

print(os.path.abspath("./legal-advisor-final-1"))
print(os.listdir("./legal-advisor-final-1"))

/content/legal-advisor-final-1
['training_args.bin', 'README.md', 'tokenizer_config.json', 'tokenizer.json', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja']


In [ ]:
import shutil

shutil.make_archive(
    "/content/legal-advisor-final-1",
    "zip",
    "/content",
    "legal-advisor-final-1"
)

print("Model folder zipped successfully!")

Model folder zipped successfully!


In [ ]:
print(formatted_dataset[0])

{'article': 0, 'title': 'Preamble', 'description': 'WE, THE PEOPLE OF INDIA, having solemnly resolved to constitute India into a SOVEREIGN SOCIALIST SECULAR DEMOCRATIC REPUBLIC and to secure to all its citizens: JUSTICE, social, economic and political; LIBERTY of thought, expression, belief, faith and worship;\nEQUALITY of status and of opportunity; and to promote among them all FRATERNITY assuring the dignity of the individual and the unity and integrity of the Nation; IN OUR CONSTITUENT ASSEMBLY this twenty-sixth day of November, 1949, do HEREBY ADOPT, ENACT AND GIVE TO OURSELVES THIS CONSTITUTION', 'text': '### Question:\nExplain the Preamble of the Constitution of India.\n\n### Answer:\nWE, THE PEOPLE OF INDIA, having solemnly resolved to constitute India into a SOVEREIGN SOCIALIST SECULAR DEMOCRATIC REPUBLIC and to secure to all its citizens: JUSTICE, social, economic and political; LIBERTY of thought, expression, belief, faith and worship;\nEQUALITY of status and of opportunity; 

In [ ]:
sample = formatted_dataset[0]["text"]

print("Ends with EOS:", sample.endswith(tokenizer.eos_token))
print("Last 100 characters:", repr(sample[-100:]))

Ends with EOS: False
Last 100 characters: 's twenty-sixth day of November, 1949, do HEREBY ADOPT, ENACT AND GIVE TO OURSELVES THIS CONSTITUTION'


In [ ]:
print("EOS token:", repr(tokenizer.eos_token))
print("EOS token ID:", tokenizer.eos_token_id)

print("\nTrainer config:")
print("EOS token setting:", getattr(new_trainer.args, "eos_token", "Not configured"))
print("Max sequence length:", getattr(new_trainer.args, "max_length", "Not configured"))

EOS token: '<|im_end|>'
EOS token ID: 151645

Trainer config:
EOS token setting: None
Max sequence length: 1024


In [ ]:
from trl import SFTConfig
import inspect

print("TRL SFTConfig parameters:")
print(inspect.signature(SFTConfig))

print("\nTrainer dataset sample:")
print(repr(new_trainer.train_dataset[0]["text"][-120:]))

print("\nTokenizer EOS token:")
print(repr(tokenizer.eos_token))

TRL SFTConfig parameters:
(output_dir: str | None = None, per_device_train_batch_size: int = 8, num_train_epochs: float = 3.0, max_steps: int = -1, learning_rate: float = 2e-05, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_steps: float = 0, optim: transformers.training_args.OptimizerNames | str = 'adamw_torch_fused', optim_args: str | None = None, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, optim_target_modules: None | str | list[str] = None, gradient_accumulation_steps: int = 1, average_tokens_across_devices: bool = True, max_grad_norm: float = 1.0, label_smoothing_factor: float = 0.0, bf16: bool | None = None, fp16: bool = False, bf16_full_eval: bool = False, fp16_full_eval: bool = False, tf32: bool | None = None, gradient_checkpointing: bool = True, gradient_checkpointing_kwargs: dict[str, typing.Any] | str | None = None, torch_compil